# Laboratorio 1 — RAG desde cero

**Inteligencia Artificial · Unidad 2, Sesión 4 · Universidad San Sebastián**

Dr. Carlos Hernández Ulloa · Reiner Romero Garrido

---

Al terminar este cuaderno habrás escrito **el bucle completo de la lámina 38**, sin ninguna biblioteca de RAG de por medio: incrustar los documentos, comparar por coseno, quedarte con los k mejores, componer el prompt y preguntar.

Son unas treinta líneas. Y lo que vas a medir es que **la calidad de la respuesta no se decide en el modelo de lenguaje**: se decide antes, en cómo partiste el documento y en qué le pusiste adelante.

### Cómo está armado

| Celda | Qué haces |
|---|---|
| **DEMO** | La ejecutas y la lees. **No la modificas.** |
| **TU TURNO** | Aquí escribes tú, donde dice `REEMPLAZA` |
| **VERIFICACIÓN** | La ejecutas y te responde sola |

**Regla de nombres:** todo lo del profesor termina en `_demo`, todo lo tuyo en `_mio` / `_mia`. En Colab las celdas comparten memoria: si reutilizas un nombre `_demo`, destruyes el ejemplo con el que hay que compararse.

### Antes de ejecutar nada

> **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**

El cambio **reinicia el cuaderno**, así que hazlo ahora y no a la mitad.

---
## Bloque 0 — La GPU, el corpus y el codificador

Ejecuta las celdas. No hay ejercicio en este bloque.

In [70]:
# === DEMO — si esta celda falla, activa la GPU y vuelve a empezar ===
import time, torch

T_INICIO = time.time()          # para el presupuesto de reloj del final

assert torch.cuda.is_available(), (
    "No hay GPU. Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU. "
    "Ojo: el cambio reinicia el cuaderno, así que hazlo antes de avanzar.")

dispositivo = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [71]:
# === DEMO — el codificador de oraciones ===
# Descarga unos 471 MB la primera vez. Después queda en caché.
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

t0 = time.time()
codificador_demo = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
SEGUNDOS_CODIFICADOR = time.time() - t0

print("dimensión del embedding:", codificador_demo.encode("passage: prueba").shape[0])
print(f"descargar y cargar el codificador: {SEGUNDOS_CODIFICADOR:.1f} s")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

dimensión del embedding: 384
descargar y cargar el codificador: 8.3 s


**Este modelo pide dos prefijos y no son un adorno:** `"query: "` para las preguntas y `"passage: "` para los textos del corpus.

Se entrenó de forma **asimétrica**. Una pregunta y el párrafo que la contesta *no son el mismo tipo de objeto*: la pregunta es corta, incompleta y en otro registro. El prefijo es lo que le avisa al modelo cuál de los dos está viendo.

Si se te olvida uno de los dos, nada se rompe — solo empeoran los resultados en silencio. Es el error más difícil de pillar de todo el cuaderno.

> **Si te salió una advertencia roja** que dice `You are sending unauthenticated requests to the HF Hub`, ignórala. Es Hugging Face avisando que estás descargando sin cuenta, cosa que está permitida y funciona. No es un error.

In [72]:
# === DEMO — el corpus: cinco documentos de Pacific-Blue ===
CORPUS_DEMO = {

"politica_despacho_2026": """Politica de Despacho 2026 - Pacific-Blue
Vigente desde el 2 de enero de 2026. Reemplaza toda instruccion anterior.

Dias de operacion. El centro de distribucion opera de lunes a viernes. No se
realizan despachos los sabados, domingos ni festivos. Un pedido confirmado un
viernes despues de las 16:00 se procesa el lunes siguiente.

Plazos comprometidos. Para la familia de mallas, el plazo minimo comprometido
al cliente es de 72 horas habiles contadas desde la confirmacion del pedido. El
area comercial no puede ofrecer un plazo menor sin autorizacion escrita de
operaciones.

Ventana de corte. Los pedidos ingresados antes de las 12:00 entran a la
programacion del mismo dia. Los posteriores entran a la del dia siguiente.

Pedidos parciales. Cuando el stock disponible no cubre la cantidad solicitada,
se despacha lo disponible y el saldo queda en espera. No se retiene el pedido
completo a la espera de reposicion.""",

"ficha_malla_p50": """Ficha Tecnica - Malla P50
Codigo de material: MP-050. Familia: mallas.

Descripcion. Malla de acero galvanizado de trama cuadrada, disenada para cierre
perimetral liviano en faenas y obras menores.

Dimensiones. Rollo de 25 metros de largo por 1,50 metros de alto. Abertura de
trama de 50 milimetros. Calibre del alambre 2,5 milimetros.

Peso. 18 kilogramos por rollo.

Existencias. Al cierre del mes hay 320 rollos disponibles en el centro de
distribucion de Coronel.

Plazo de entrega. 72 horas habiles desde la confirmacion del pedido, segun la
politica vigente.""",

"ficha_malla_p80": """Ficha Tecnica - Malla P80
Codigo de material: MP-080. Familia: mallas.

Descripcion. Malla de acero galvanizado de trama cuadrada, disenada para cierre
perimetral de alta exigencia en faenas industriales.

Dimensiones. Rollo de 25 metros de largo por 2,00 metros de alto. Abertura de
trama de 80 milimetros. Calibre del alambre 3,5 milimetros.

Peso. 41 kilogramos por rollo.

Existencias. Al cierre del mes hay 45 rollos disponibles en el centro de
distribucion de Coronel.

Plazo de entrega. 96 horas habiles desde la confirmacion del pedido, por
tratarse de un producto de fabricacion programada.""",

"tarifario_2026": """Tarifario de Transporte 2026 - Pacific-Blue
Valores en pesos por rollo, sin impuestos incluidos, vigentes al 1 de marzo. La
tarifa se aplica por rollo despachado y no por pedido. El plazo adicional se
suma al plazo comprometido del producto en su ficha tecnica.

Destino            Plazo adicional   Tarifa por rollo   Carga minima
Coronel                 0 dias              2.400            1 rollo
Concepcion              0 dias              3.100            1 rollo
Lota                    0 dias              2.900            1 rollo
Los Angeles             1 dia               5.800            4 rollos
Chillan                 1 dia               6.700            6 rollos
Temuco                  2 dias              9.400            8 rollos
Valdivia                2 dias             10.100            8 rollos
Santiago                2 dias             11.200           10 rollos
Antofagasta             4 dias             19.800           20 rollos

Recargos. Los envios a destinos fuera de la Region del Biobio con menos de la
carga minima pagan un recargo de 40 por ciento sobre la tarifa por rollo.

Retiro en planta. El cliente que retira en el centro de distribucion de Coronel
no paga tarifa de transporte.""",

"preguntas_frecuentes": """Preguntas Frecuentes - Pacific-Blue

Como pido una cotizacion. Escribiendo al ejecutivo comercial asignado a su
cuenta. La cotizacion tiene una validez de 15 dias corridos.

Que pasa si el material llega danado. Hay que informarlo dentro de las 48 horas
siguientes a la recepcion, con fotografias del embalaje. El reemplazo se
despacha sin costo de transporte.

Puedo cambiar un producto por otro. Se aceptan cambios de material dentro de los
10 dias corridos siguientes a la entrega, siempre que el rollo este sin abrir.

Emiten certificado de calidad. Si. Cada lote de mallas se entrega con su
certificado de galvanizado, disponible tambien en formato digital.""",
}

print(f"{len(CORPUS_DEMO)} documentos · {sum(len(t) for t in CORPUS_DEMO.values()):,} caracteres")
for nombre, texto in CORPUS_DEMO.items():
    print(f"  {nombre:24s} {len(texto):5,d} car   {texto.splitlines()[0]}")

5 documentos · 3,967 caracteres
  politica_despacho_2026     919 car   Politica de Despacho 2026 - Pacific-Blue
  ficha_malla_p50            565 car   Ficha Tecnica - Malla P50
  ficha_malla_p80            599 car   Ficha Tecnica - Malla P80
  tarifario_2026           1,222 car   Tarifario de Transporte 2026 - Pacific-Blue
  preguntas_frecuentes       662 car   Preguntas Frecuentes - Pacific-Blue


**Léelo entero antes de seguir.** Son cuatro mil caracteres y todo lo que midas después depende de saber qué dice y qué no dice.

Dos cosas para tener presentes:

- La **P50** y la **P80** se parecen muchísimo en el texto y difieren en cada número.
- Hay temas que **ningún documento cubre**. Anda pensando cuáles.

In [73]:
# === DEMO — el corpus completo, para leerlo ===
for nombre, texto in CORPUS_DEMO.items():
    print("=" * 78)
    print(texto)

Politica de Despacho 2026 - Pacific-Blue
Vigente desde el 2 de enero de 2026. Reemplaza toda instruccion anterior.

Dias de operacion. El centro de distribucion opera de lunes a viernes. No se
realizan despachos los sabados, domingos ni festivos. Un pedido confirmado un
viernes despues de las 16:00 se procesa el lunes siguiente.

Plazos comprometidos. Para la familia de mallas, el plazo minimo comprometido
al cliente es de 72 horas habiles contadas desde la confirmacion del pedido. El
area comercial no puede ofrecer un plazo menor sin autorizacion escrita de
operaciones.

Ventana de corte. Los pedidos ingresados antes de las 12:00 entran a la
programacion del mismo dia. Los posteriores entran a la del dia siguiente.

Pedidos parciales. Cuando el stock disponible no cubre la cantidad solicitada,
se despacha lo disponible y el saldo queda en espera. No se retiene el pedido
completo a la espera de reposicion.
Ficha Tecnica - Malla P50
Codigo de material: MP-050. Familia: mallas.

Descripc

---
## Bloque 1 — De texto a vector, y el coseno a mano

*Objetivo: que el embedding de oración deje de ser una caja.*

In [74]:
# === DEMO — un texto entra, 384 números salen ===
import numpy as np

frases_demo = [
    "El plazo de entrega es de 72 horas habiles.",
    "Despachamos en tres dias habiles.",
    "El rollo pesa 18 kilogramos.",
]
vectores_demo = codificador_demo.encode(["passage: " + f for f in frases_demo])

print("forma:", vectores_demo.shape)      # (3, 384): un vector por frase
print("primeros 6 números de la frase 1:", np.round(vectores_demo[0][:6], 3))

forma: (3, 384)
primeros 6 números de la frase 1: [-0.019 -0.009 -0.001 -0.096  0.059 -0.056]


**Tres frases, tres vectores de 384 números.** No importa que una tenga cuarenta caracteres y otra ochenta: el embedding de oración siempre entrega un vector del mismo largo.

Ojo con la diferencia respecto de la lámina 24: el embedding **contextual** da un vector *por token*; éste da uno *por oración*. Son dos objetos distintos y acá usamos el segundo.

**TU TURNO**

Escribe el coseno con la fórmula de la lámina 37:

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

Sin `sklearn`, sin `util.cos_sim`: `np.dot` y `np.linalg.norm`.

In [75]:
# === TU TURNO ===
def coseno_mio(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))    # <-- REEMPLAZA por la fórmula


print("frase 1 consigo misma:", coseno_mio(vectores_demo[0], vectores_demo[0]))
print("las dos del plazo    :", coseno_mio(vectores_demo[0], vectores_demo[1]))
print("plazo contra peso    :", coseno_mio(vectores_demo[0], vectores_demo[2]))

frase 1 consigo misma: 1.0
las dos del plazo    : 0.882699
plazo contra peso    : 0.8554899


In [76]:
# === VERIFICACIÓN ===
a, b, c = vectores_demo
assert coseno_mio(a, a) is not None, "coseno_mio sigue devolviendo None"
assert abs(coseno_mio(a, a) - 1.0) < 1e-6, \
    f"El coseno de un vector consigo mismo tiene que ser 1,0 y te dio {coseno_mio(a, a)}"
assert abs(coseno_mio(a, b) - coseno_mio(b, a)) < 1e-9, "El coseno es simétrico"
assert -1.0001 <= coseno_mio(a, c) <= 1.0001, "El coseno vive entre -1 y 1"
assert coseno_mio(a, b) > coseno_mio(a, c), \
    "Las dos frases de plazos tienen que parecerse más entre sí que con la del peso"
print("Correcto: es un coseno, y ordena como corresponde.")

Correcto: es un coseno, y ordena como corresponde.


**Un detalle que vale para todo lo que viene:** si los vectores están **normalizados** (largo 1), el denominador es 1 y el coseno es el producto punto y nada más.

Por eso en el bloque 4 el índice se normaliza una vez al construirlo, y después buscar es *una multiplicación de matrices*. La fórmula completa se escribe una vez, para entenderla; después se paga menos.

---
## Bloque 2 — Buscar por palabras contra buscar por dirección

*Objetivo: comprobar la afirmación de la lámina 37, y verla fallar del otro lado.*

Primero preparamos los dos buscadores. El de palabras es el que cualquiera escribiría: cuenta cuántas palabras comparten la pregunta y el párrafo.

In [77]:
# === DEMO — los trozos de trabajo de este bloque, y el buscador por palabras ===
import re

def parrafos_con_titulo(corpus):
    """Cada párrafo, con el título de su documento pegado adelante."""
    fuera = []
    for nombre, texto in corpus.items():
        cabeza = texto.split("\n")[0]
        for p in texto.split("\n\n"):
            p = " ".join(p.split())
            if p:
                fuera.append(f"{cabeza}. {p}")
    return fuera


trozos_demo = parrafos_con_titulo(CORPUS_DEMO)
print(len(trozos_demo), "trozos")

VACIAS = {"de", "la", "el", "los", "las", "un", "una", "en", "y", "o", "a", "que", "se",
          "por", "con", "del", "al", "es", "son", "para", "cuanto", "cuantos", "cuanta",
          "cuantas", "como", "hay"}

def palabras(texto):
    return {p for p in re.findall(r"[a-z0-9áéíóúñ]+", texto.lower()) if p not in VACIAS}


def buscar_por_palabras(pregunta):
    pp = palabras(pregunta)
    conteos = [len(pp & palabras(t)) for t in trozos_demo]
    i = max(range(len(trozos_demo)), key=lambda j: conteos[j])
    return trozos_demo[i], conteos[i]

26 trozos


In [78]:
# === DEMO — el índice, y los dos buscadores frente a frente ===
indice_demo = codificador_demo.encode(["passage: " + t for t in trozos_demo],
                                      convert_to_tensor=True, normalize_embeddings=True)

def buscar_por_coseno(pregunta):
    v = codificador_demo.encode("query: " + pregunta, convert_to_tensor=True,
                                normalize_embeddings=True)
    puntajes = indice_demo @ v
    i = int(torch.argmax(puntajes))
    return trozos_demo[i], float(puntajes[i])


for pregunta in ["cuanto se demoran en despachar", "atienden los fines de semana"]:
    t_pal, n = buscar_por_palabras(pregunta)
    t_cos, s = buscar_por_coseno(pregunta)
    print(f"\n### {pregunta!r}")
    print(f"  por palabras ({n} en común): {t_pal[:88]}")
    print(f"  por coseno   ({s:.3f})      : {t_cos[:88]}")
    print(f"  la pregunta y el párrafo del coseno comparten: "
          f"{sorted(palabras(pregunta) & palabras(t_cos)) or 'NINGUNA PALABRA'}")


### 'cuanto se demoran en despachar'
  por palabras (0 en común): Politica de Despacho 2026 - Pacific-Blue. Politica de Despacho 2026 - Pacific-Blue Vigen
  por coseno   (0.875)      : Politica de Despacho 2026 - Pacific-Blue. Plazos comprometidos. Para la familia de malla
  la pregunta y el párrafo del coseno comparten: NINGUNA PALABRA

### 'atienden los fines de semana'
  por palabras (0 en común): Politica de Despacho 2026 - Pacific-Blue. Politica de Despacho 2026 - Pacific-Blue Vigen
  por coseno   (0.827)      : Politica de Despacho 2026 - Pacific-Blue. Dias de operacion. El centro de distribucion o
  la pregunta y el párrafo del coseno comparten: NINGUNA PALABRA


**Mira la última línea de cada bloque de salida.**

Las dos preguntas encuentran el párrafo que las contesta **sin compartir ni una sola palabra de contenido** con él. La política nunca dice «cuánto se demoran»: dice *«plazos comprometidos»*. Nunca dice «fin de semana»: dice *«sábados, domingos ni festivos»*.

El buscador por palabras, mientras tanto, no tiene con qué: cero coincidencias, y devuelve cualquier cosa.

**Eso es la sesión completa.** No se busca por palabras, se busca por dirección en el espacio.

**TU TURNO**

Escribe **dos preguntas tuyas** sobre el corpus que cumplan las dos condiciones:

1. Que el corpus sí las conteste.
2. Que **no compartan ninguna palabra de contenido** con el párrafo que las contesta.

Tendrás que leer el corpus para lograrlo. Ése es el punto.

In [79]:
# === TU TURNO ===
mis_preguntas = [
    "Que ocurre frente a mercaderia rota",    # <-- REEMPLAZA
    "Cobran flete por buscar la compra uno mismo en bodega",    # <-- REEMPLAZA
]

for pregunta in mis_preguntas:
    trozo, s = buscar_por_coseno(pregunta)
    compartidas = sorted(palabras(pregunta) & palabras(trozo))
    print(f"\n{pregunta!r}  ->  coseno {s:.3f}")
    print(f"  {trozo[:95]}")
    print(f"  comparten: {compartidas or 'NINGUNA PALABRA'}")


'Que ocurre frente a mercaderia rota'  ->  coseno 0.821
  Tarifario de Transporte 2026 - Pacific-Blue. Retiro en planta. El cliente que retira en el cent
  comparten: NINGUNA PALABRA

'Cobran flete por buscar la compra uno mismo en bodega'  ->  coseno 0.810
  Politica de Despacho 2026 - Pacific-Blue. Ventana de corte. Los pedidos ingresados antes de las
  comparten: ['mismo']


In [80]:
# === VERIFICACIÓN ===
assert all(p.strip() for p in mis_preguntas), "Faltan tus dos preguntas"

for pregunta in ["cuanto se demoran en despachar", "atienden los fines de semana"]:
    t_pal, _ = buscar_por_palabras(pregunta)
    t_cos, _ = buscar_por_coseno(pregunta)
    assert t_pal != t_cos, \
        f"Los dos buscadores devolvieron el mismo párrafo para {pregunta!r}"
    assert not (palabras(pregunta) & palabras(t_cos)), \
        f"{pregunta!r} sí comparte palabras con el párrafo que encontró"

sin_palabras = [p for p in mis_preguntas if not (palabras(p) & palabras(buscar_por_coseno(p)[0]))]
assert sin_palabras, (
    "Ninguna de tus dos preguntas logró cero palabras en común. Reescríbelas: "
    "di lo mismo con otras palabras que las del documento.")
print(f"Correcto: {len(sin_palabras)} de tus 2 preguntas encontraron su párrafo sin "
      f"compartir ninguna palabra. El coseno ve lo que la búsqueda por palabras no puede ver.")

Correcto: 1 de tus 2 preguntas encontraron su párrafo sin compartir ninguna palabra. El coseno ve lo que la búsqueda por palabras no puede ver.


---
## Bloque 3 — Trocear: cuatro maneras, medidas

*Objetivo: el de la lámina 35 — que el troceo es la decisión más subestimada de todo el RAG.*

Para poder comparar hace falta una regla. Estas son **nueve preguntas con respuesta conocida**: cada una viene con el fragmento de texto que el trozo recuperado tiene que contener.

In [81]:
# === DEMO — la regla de medición ===
PRUEBAS_DEMO = [
    ("cuanto se demoran en despachar una malla P50",              "72 horas"),
    ("cuantos rollos de malla P50 hay disponibles",               "320 rollos"),
    ("cuantos rollos de malla P80 hay disponibles",               "45 rollos"),
    ("cuanto pesa un rollo de la P80",                            "41 kilogramos"),
    ("despachan los fines de semana",                             "No se realizan despachos"),
    ("si confirmo un pedido el viernes en la tarde, cuando sale", "16:00"),
    ("cuanto cuesta llevar rollos hasta Temuco",                  "9.400"),
    ("cuantos rollos como minimo para enviar a Santiago",         "10 rollos"),
    ("me sirve el material si llega roto",                        "48 horas"),
]

def evaluar(trozos, k=4, mostrar=False):
    """Cuántas de las nueve preguntas traen su respuesta en el trozo número 1."""
    indice = codificador_demo.encode(["passage: " + t for t in trozos],
                                     convert_to_tensor=True, normalize_embeddings=True)
    top1 = 0
    for pregunta, fragmento in PRUEBAS_DEMO:
        v = codificador_demo.encode("query: " + pregunta, convert_to_tensor=True,
                                    normalize_embeddings=True)
        mejores = torch.topk(indice @ v, min(k, len(trozos)))
        primero = trozos[int(mejores.indices[0])]
        acierta = fragmento in primero
        top1 += acierta
        if mostrar and not acierta:
            print(f"    FALLA {pregunta!r}\n          -> {primero[:88]}")
    return top1


print(f"{len(PRUEBAS_DEMO)} preguntas de prueba, cada una con el fragmento que su respuesta necesita")

9 preguntas de prueba, cada una con el fragmento que su respuesta necesita


In [82]:
# === DEMO — tres maneras de partir el mismo corpus ===
def por_documento(corpus):
    return [" ".join(t.split()) for t in corpus.values()]


def a_lo_bruto(corpus, n_car=500):
    fuera = []
    for texto in corpus.values():
        plano = " ".join(texto.split())
        for i in range(0, len(plano), n_car):
            fuera.append(plano[i:i + n_car])
    return fuera


def por_parrafo(corpus):
    fuera = []
    for texto in corpus.values():
        for p in texto.split("\n\n"):
            p = " ".join(p.split())
            if p:
                fuera.append(p)
    return fuera


for nombre, funcion in [("A · un vector por documento", por_documento),
                        ("B · cada 500 caracteres, a lo bruto", a_lo_bruto),
                        ("C · por párrafo", por_parrafo)]:
    trozos = funcion(CORPUS_DEMO)
    aciertos = evaluar(trozos)
    print(f"{nombre:38s} {len(trozos):3d} trozos, {sum(map(len, trozos))//len(trozos):4d} car medios"
          f"  ->  {aciertos}/9")

A · un vector por documento              5 trozos,  716 car medios  ->  9/9
B · cada 500 caracteres, a lo bruto     10 trozos,  358 car medios  ->  8/9
C · por párrafo                         26 trozos,  137 car medios  ->  5/9


**Detente en el resultado de C.**

Partir por párrafo es lo que suena más razonable — lo dice la propia lámina 35 — y es **el peor de los tres**. Con cuatro veces más trozos, acierta menos.

La celda siguiente muestra por qué.

In [83]:
# === DEMO — el trozo culpable ===
print("Los párrafos que fallan, tal como quedan al partir por párrafo:\n")
for t in por_parrafo(CORPUS_DEMO):
    if "Existencias" in t or t.startswith("Peso"):
        print(" ", t)

print()
evaluar(por_parrafo(CORPUS_DEMO), mostrar=True)

Los párrafos que fallan, tal como quedan al partir por párrafo:

  Peso. 18 kilogramos por rollo.
  Existencias. Al cierre del mes hay 320 rollos disponibles en el centro de distribucion de Coronel.
  Peso. 41 kilogramos por rollo.
  Existencias. Al cierre del mes hay 45 rollos disponibles en el centro de distribucion de Coronel.

    FALLA 'cuanto se demoran en despachar una malla P50'
          -> Que pasa si el material llega danado. Hay que informarlo dentro de las 48 horas siguient
    FALLA 'cuantos rollos de malla P50 hay disponibles'
          -> Ficha Tecnica - Malla P50 Codigo de material: MP-050. Familia: mallas.
    FALLA 'cuantos rollos de malla P80 hay disponibles'
          -> Ficha Tecnica - Malla P80 Codigo de material: MP-080. Familia: mallas.
    FALLA 'cuanto pesa un rollo de la P80'
          -> Peso. 18 kilogramos por rollo.


5

**Ese párrafo no dice de qué producto habla.**

```
Existencias. Al cierre del mes hay 320 rollos disponibles en el centro de distribucion de Coronel.
```

El título del documento lo decía —*Ficha Técnica - Malla P50*— y el troceo lo botó. Preguntando por el stock de la P50, el coseno devuelve el párrafo de existencias de la **P80**, que dice exactamente lo mismo con otro número.

**La regla que hay que llevarse:** *un trozo tiene que poder leerse solo.*

**TU TURNO**

Arréglalo. Escribe `por_parrafo_con_titulo_mia`: lo mismo que `por_parrafo`, pero pegándole a cada trozo **el título de su documento** (la primera línea) adelante.

Antes de ejecutar: **¿cuánto crees que sube el acierto?**

In [84]:
# === TU TURNO ===
def por_parrafo_con_titulo_mia(corpus):
    fuera = []
    for texto in corpus.values():
        cabeza = texto.split("\n")[0]                          # <-- REEMPLAZA: la primera línea del documento
        for p in texto.split("\n\n"):
            p = " ".join(p.split())
            if p:
                fuera.append(f"{cabeza}. {p}")                # <-- REEMPLAZA: pégale el título adelante
    return fuera


trozos_mios = por_parrafo_con_titulo_mia(CORPUS_DEMO)
print(len(trozos_mios), "trozos ·", trozos_mios[1][:95])
print("aciertos:", evaluar(trozos_mios), "/ 9   (por párrafo pelado:", evaluar(por_parrafo(CORPUS_DEMO)), "/ 9)")

26 trozos · Politica de Despacho 2026 - Pacific-Blue. Dias de operacion. El centro de distribucion opera de
aciertos: 9 / 9   (por párrafo pelado: 5 / 9)


In [85]:
# === VERIFICACIÓN ===
assert len(trozos_mios) == 26, f"Se esperaban 26 trozos y tienes {len(trozos_mios)}"
assert all("Pacific-Blue" in t or "Ficha Tecnica" in t for t in trozos_mios), \
    "Hay trozos sin el título de su documento"

aciertos_mios = evaluar(trozos_mios)
aciertos_pelado = evaluar(por_parrafo(CORPUS_DEMO))
assert aciertos_mios - aciertos_pelado >= 3, \
    f"El título tiene que ganar al menos 3 aciertos: pasaste de {aciertos_pelado} a {aciertos_mios}"
print(f"Correcto: de {aciertos_pelado}/9 a {aciertos_mios}/9 pegando una línea.")

Correcto: de 5/9 a 9/9 pegando una línea.


### Entonces, ¿para qué trocear?

Porque el troceo por documento entero también da 9 de 9. Con cinco documentos, claro que sí: el corpus completo cabe en la ventana de contexto.

La celda siguiente mide lo que **cuesta** cada opción.

In [86]:
# === DEMO — lo que cuesta el contexto ===
!pip install -q transformers accelerate
from transformers import AutoTokenizer

t0 = time.time()
tok_demo = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
SEGUNDOS_TOKENIZADOR = time.time() - t0
print(f"descargar y cargar el tokenizador: {SEGUNDOS_TOKENIZADOR:.1f} s\n")

def tokens_de_contexto(trozos, k=4):
    indice = codificador_demo.encode(["passage: " + t for t in trozos],
                                     convert_to_tensor=True, normalize_embeddings=True)
    total = []
    for pregunta, _ in PRUEBAS_DEMO:
        v = codificador_demo.encode("query: " + pregunta, convert_to_tensor=True,
                                    normalize_embeddings=True)
        mejores = torch.topk(indice @ v, min(k, len(trozos)))
        contexto = "\n\n".join(trozos[int(i)] for i in mejores.indices)
        total.append(len(tok_demo(contexto)["input_ids"]))
    return round(sum(total) / len(total))


print(f"{'troceo':40s} {'trozos':>7s} {'tokens':>8s} {'aciertos':>9s}")
for nombre, trozos in [("A · un vector por documento", por_documento(CORPUS_DEMO)),
                       ("B · cada 500 caracteres", a_lo_bruto(CORPUS_DEMO)),
                       ("C · por párrafo", por_parrafo(CORPUS_DEMO)),
                       ("D · por párrafo con título (el tuyo)", trozos_mios)]:
    print(f"{nombre:40s} {len(trozos):7d} {tokens_de_contexto(trozos):8d} {evaluar(trozos):8d}/9")

descargar y cargar el tokenizador: 1.4 s

troceo                                    trozos   tokens  aciertos
A · un vector por documento                    5      829        9/9
B · cada 500 caracteres                       10      518        8/9
C · por párrafo                               26      176        5/9
D · por párrafo con título (el tuyo)          26      219        9/9


**Ahí está la respuesta.** Tu troceo y el de documento entero aciertan lo mismo, pero el tuyo lo hace con **casi cuatro veces menos contexto**.

Con cinco documentos da igual. Con el manual completo de una empresa no cabe, y por la lámina 23 **cada token se paga al cuadrado**.

> **Se trocea para pagar menos contexto, no para acertar más.** Y eso solo se ve mirando las dos columnas juntas.

In [87]:
# === DEMO — lo que el corte a lo bruto le hizo a la tabla del tarifario ===
trozos_brutos = a_lo_bruto({"tarifario_2026": CORPUS_DEMO["tarifario_2026"]})
print("final del primer trozo:")
print("   ...", trozos_brutos[0][-70:])
print("\ninicio del segundo:")
print("   ", trozos_brutos[1][:70], "...")

final del primer trozo:
   ... ollos Chillan 1 dia 6.700 6 rollos Temuco 2 dias 9.400 8 rollos Valdiv

inicio del segundo:
    ia 2 dias 10.100 8 rollos Santiago 2 dias 11.200 10 rollos Antofagasta ...


**La fila de Valdivia quedó partida en dos trozos, con el nombre cortado a la mitad:** `Valdiv` | `ia 2 dias 10.100`.

Ningún indicador del sistema se entera de esto. El error aparece recién en la respuesta, cuando alguien pregunte por Valdivia y el modelo conteste con seguridad usando media tabla.

Es el tercer modo de falla de la lámina 40 —*el trozo mal leído*— y lo vas a provocar a propósito en el Laboratorio 2.

---
## Bloque 4 — El índice y `buscar(pregunta, k)`

*Objetivo: las láminas 36 y 37, puestas en una función.*

El índice es la lámina 36: cada trozo pasa por el codificador **una sola vez** y queda guardado como un vector.

In [88]:
# === DEMO — el índice sobre tu troceo ===
indice_mio = codificador_demo.encode(["passage: " + t for t in trozos_mios],
                                     convert_to_tensor=True, normalize_embeddings=True)
print("forma del índice:", tuple(indice_mio.shape))   # (26, 384)
print("¿está normalizado?", float(indice_mio[0].norm()))

forma del índice: (26, 384)
¿está normalizado? 1.0


**TU TURNO**

Escribe `buscar_mia`. Con los vectores ya normalizados son tres líneas: incrustar la pregunta, multiplicar por el índice, quedarse con los k mayores.

Devuelve una lista de pares `(coseno, trozo)`, del más parecido al menos.

In [89]:
# === TU TURNO ===
def buscar_mia(pregunta, k=4):
    v = codificador_demo.encode("query: " + pregunta, convert_to_tensor=True,
                                normalize_embeddings=True)
    puntajes = indice_mio @ v                    # <-- REEMPLAZA: el coseno contra todo el índice
    mejores = torch.topk(puntajes, min(k, len(trozos_mios)))                     # <-- REEMPLAZA: los k mayores (mira torch.topk)
    return [(float(s), trozos_mios[int(i)]) for s, i in zip(mejores.values, mejores.indices)]


for s, t in buscar_mia("cuantos rollos de malla P50 hay disponibles"):
    print(f"{s:.4f}  {t[:88]}")

0.9320  Ficha Tecnica - Malla P50. Existencias. Al cierre del mes hay 320 rollos disponibles en 
0.9128  Ficha Tecnica - Malla P50. Peso. 18 kilogramos por rollo.
0.9111  Ficha Tecnica - Malla P80. Existencias. Al cierre del mes hay 45 rollos disponibles en e
0.9103  Ficha Tecnica - Malla P50. Dimensiones. Rollo de 25 metros de largo por 1,50 metros de a


In [90]:
# === VERIFICACIÓN ===
r = buscar_mia("cuantos rollos de malla P50 hay disponibles", k=4)
assert r is not None and len(r) == 4, "buscar_mia tiene que devolver exactamente k resultados"
puntajes = [s for s, _ in r]
assert puntajes == sorted(puntajes, reverse=True), "Los resultados vienen del mejor al peor"
assert "320 rollos" in r[0][1], "El primer trozo debería traer los 320 rollos de la P50"

esperados = {"cuantos rollos de malla P50 hay disponibles": 0.932,
             "cuantos rollos de malla P80 hay disponibles": 0.928,
             "despachan los fines de semana": 0.870}
for pregunta, valor in esperados.items():
    obtenido = buscar_mia(pregunta, k=1)[0][0]
    assert abs(obtenido - valor) < 0.005, \
        f"Para {pregunta!r} se esperaba un coseno de {valor} y salió {obtenido:.3f}"
print("Correcto: los tres cosenos de referencia coinciden.")

Correcto: los tres cosenos de referencia coinciden.


**El margen con el que acertó la primera:**

El párrafo correcto saca **0,9320**. El de existencias de la P80 —que dice lo mismo con otro número— saca **0,9111**. Veintiún milésimas, y en medio se cuela el párrafo del peso de la P50.

Acertó. Y el margen con el que acertó es ése.

### Cuántos trozos hay que traer

Con las nueve preguntas de antes, `k=1` ya acierta las nueve: cada una se contesta con **un** párrafo. Las preguntas reales no son así.

Estas cinco necesitan **dos trozos distintos** cada una, y por eso sirven para elegir `k`.

In [91]:
# === DEMO — preguntas que necesitan más de un trozo ===
PRUEBAS_DOBLES = [
    ("cual de las dos mallas pesa mas, la P50 o la P80",            ["18 kilogramos", "41 kilogramos"]),
    ("cuanto cuesta mandar rollos de P80 a Temuco y cuanto demora", ["9.400", "96 horas"]),
    ("me pueden despachar 400 mallas P50 el sabado",                ["320 rollos", "No se realizan despachos"]),
    ("cuantos rollos de P50 hay y cuanto pesa cada uno",            ["320 rollos", "18 kilogramos"]),
    ("que malla es mas alta y cuantos rollos quedan de ella",       ["2,00 metros", "45 rollos"]),
]
print(len(PRUEBAS_DOBLES), "preguntas, cada una con los dos datos que su respuesta necesita")

5 preguntas, cada una con los dos datos que su respuesta necesita


**TU TURNO**

Mueve `k` y mira las dos columnas moverse en direcciones opuestas. Cuenta una pregunta solo si **los dos** fragmentos quedaron dentro del contexto.

In [92]:
# === TU TURNO ===
for k in [1, 2, 4, 8, 12]:
    completas = 0
    tokens = 0
    for pregunta, fragmentos in PRUEBAS_DOBLES:
        trozos = [t for _, t in buscar_mia(pregunta, k=k)]                              # <-- REEMPLAZA: usa buscar_mia
        contexto = "\n\n".join(trozos)                            # <-- REEMPLAZA: únelos con "\n\n"
        completas += all(f in contexto for f in fragmentos)
        tokens += len(tok_demo(contexto)["input_ids"])
    print(f"k={k:2d}:  {completas}/5 preguntas con TODOS sus datos en el contexto"
          f"  ·  {tokens // len(PRUEBAS_DOBLES):4d} tokens de contexto")

k= 1:  0/5 preguntas con TODOS sus datos en el contexto  ·    26 tokens de contexto
k= 2:  2/5 preguntas con TODOS sus datos en el contexto  ·    65 tokens de contexto
k= 4:  2/5 preguntas con TODOS sus datos en el contexto  ·   149 tokens de contexto
k= 8:  4/5 preguntas con TODOS sus datos en el contexto  ·   369 tokens de contexto
k=12:  5/5 preguntas con TODOS sus datos en el contexto  ·   565 tokens de contexto


**Ahora sí hay una decisión que tomar.** Con `k=4` solo dos de las cinco preguntas tienen todo lo que necesitan; para llegar a las cinco hay que subir a `k=12` y pagar casi cuatro veces más contexto.

**No hay un k correcto**, y eso en un proyecto es una decisión de presupuesto que se toma con esta tabla a la vista. Guárdala: la última pregunta del cuaderno es una de estas cinco.

---
## Bloque 5 — Cerrar el bucle

Recién ahora aparece el modelo de lenguaje, en el minuto 56 de 80.

**Es a propósito.** Cuatro quintos del laboratorio ocurrieron antes de que el generador existiera, y ésa es la proporción correcta: todo lo que decide la calidad de la respuesta ya pasó.

In [93]:
# === DEMO — el generador ===
# Descarga unos 3,1 GB la primera vez.
from transformers import AutoModelForCausalLM

t0 = time.time()
generador_demo = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", dtype=torch.float16, device_map="cuda")
generador_demo.eval()
SEGUNDOS_GENERADOR = time.time() - t0

n_par = sum(p.numel() for p in generador_demo.parameters())
print(f"{n_par/1e9:.2f} mil millones de parámetros")
print(f"descargar y cargar el generador: {SEGUNDOS_GENERADOR:.1f} s")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

1.54 mil millones de parámetros
descargar y cargar el generador: 15.0 s


Mil quinientos millones de parámetros: la lámina 29 en la mano. Es un modelo **chico** —los de la nube andan cien veces más arriba— y alcanza de sobra, porque lo único que le vamos a pedir es que redacte con lo que le pongamos adelante.

In [94]:
# === DEMO — la plantilla, el prompt y la respuesta ===
PLANTILLA_DEMO = """Responde SOLO con la informacion del CONTEXTO.
Si la respuesta no esta en el contexto, di exactamente: No lo se con los documentos que tengo.
Responde en una o dos frases.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}"""


SEGUNDOS_GENERACION = []      # cada llamada deja acá cuánto tardó

def preguntarle_al_modelo(prompt, max_tokens=90):
    mensajes = [{"role": "user", "content": prompt}]
    texto = tok_demo.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=False)
    entrada = tok_demo(texto, return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        salida = generador_demo.generate(**entrada, max_new_tokens=max_tokens,
                                         do_sample=False, pad_token_id=tok_demo.eos_token_id)
    SEGUNDOS_GENERACION.append(time.time() - t0)
    return tok_demo.decode(salida[0][entrada["input_ids"].shape[1]:],
                           skip_special_tokens=True).strip()


pregunta_demo = "cuantos rollos de malla P50 hay disponibles?"
contexto_demo = "\n\n".join(t for _, t in buscar_mia(pregunta_demo, k=4))
prompt_demo = PLANTILLA_DEMO.format(contexto=contexto_demo, pregunta=pregunta_demo)

print("======= EL PROMPT QUE RECIBE EL MODELO =======")
print(prompt_demo)
print("\n======= LA RESPUESTA =======")
print(preguntarle_al_modelo(prompt_demo))
print(f"\n({len(tok_demo(prompt_demo)['input_ids'])} tokens de entrada · "
      f"{SEGUNDOS_GENERACION[-1]:.1f} s en generar)")

======= EL PROMPT QUE RECIBE EL MODELO =======
Responde SOLO con la informacion del CONTEXTO.
Si la respuesta no esta en el contexto, di exactamente: No lo se con los documentos que tengo.
Responde en una o dos frases.

CONTEXTO:
Ficha Tecnica - Malla P50. Existencias. Al cierre del mes hay 320 rollos disponibles en el centro de distribucion de Coronel.

Ficha Tecnica - Malla P50. Peso. 18 kilogramos por rollo.

Ficha Tecnica - Malla P50. Dimensiones. Rollo de 25 metros de largo por 1,50 metros de alto. Abertura de trama de 50 milimetros. Calibre del alambre 2,5 milimetros.

Ficha Tecnica - Malla P80. Existencias. Al cierre del mes hay 45 rollos disponibles en el centro de distribucion de Coronel.

PREGUNTA: cuantos rollos de malla P50 hay disponibles?

======= LA RESPUESTA =======
Hay 320 rollos disponibles de malla P50 en el centro de distribución de Coronel.

(223 tokens de entrada · 2.6 s en generar)


**Lee el prompt completo antes de seguir.** No hay nada más: ni reentrenamiento, ni ajuste fino, ni memoria. El modelo es el mismo de antes y lo único que cambió es **el texto que le pusimos adelante**.

**TU TURNO**

Junta los cinco pasos de la lámina 38 en una función.

In [95]:
# === TU TURNO ===
def responder_mia(pregunta, k=4, plantilla=PLANTILLA_DEMO):
    trozos = buscar_mia(pregunta, k=k)                     # <-- REEMPLAZA: 1, 2 y 3 — buscar_mia hace los tres
    contexto = "\n\n".join(t for _, t in trozos)                   # <-- REEMPLAZA: 4 — únelos con "\n\n"
    prompt = plantilla.format(contexto=contexto, pregunta=pregunta)                     # <-- REEMPLAZA: 5 — arma el prompt con la plantilla
    return preguntarle_al_modelo(prompt)


for p in ["me pueden despachar 400 mallas P50 el sabado?",
          "cual de las dos mallas es mas pesada, la P50 o la P80?",
          "puedo pagar con factura a 30 dias?"]:
    print(f"\n### {p}\n{responder_mia(p)}")


### me pueden despachar 400 mallas P50 el sabado?
No se puede despachar 400 mallas P50 el sábado debido al plazo de entrega establecido de 72 horas habiles desde la confirmación del pedido.

### cual de las dos mallas es mas pesada, la P50 o la P80?
La malla P80 es más pesada que la P50. La P80 pesa 41 kilogramos por rollo y la P50 pesa solo 18 kilogramos por rollo.

### puedo pagar con factura a 30 dias?
No se con los documentos que tengo.


In [96]:
# === VERIFICACIÓN ===
pregunta_v = "cual de las dos mallas es mas pesada, la P50 o la P80?"
trozos_v = buscar_mia(pregunta_v, k=4)
contexto_v = "\n\n".join(t for _, t in trozos_v)
prompt_v = PLANTILLA_DEMO.format(contexto=contexto_v, pregunta=pregunta_v)

assert all(t in prompt_v for _, t in trozos_v), "El prompt tiene que traer los k trozos"
assert pregunta_v in prompt_v, "El prompt tiene que traer la pregunta"
assert "41 kilogramos" in contexto_v and "18 kilogramos" in contexto_v, \
    "Para contestar esa pregunta hacen falta los pesos de las DOS mallas en el contexto"
assert isinstance(responder_mia("cuanto pesa un rollo de la P80?"), str)
print("Correcto: el bucle está cerrado, y el contexto trae los dos documentos que hacían falta.")

Correcto: el bucle está cerrado, y el contexto trae los dos documentos que hacían falta.


**La verificación no mira lo que el modelo escribió, y no es descuido.**

Es la lámina 39: RAG garantiza que el modelo **tuvo los documentos correctos a la vista**, y nada más. No garantiza que los use, ni que los lea bien. Un `assert` sobre el texto generado estaría afirmando una garantía que el mecanismo no da.

Lo que sí se puede verificar —y es lo que hicimos— es que los trozos correctos llegaron al contexto.

**TU TURNO**

La plantilla es la superficie de control. Cámbiala por una amable, sin la prohibición de inventar, y hazle las mismas preguntas **que el corpus no contesta**.

In [97]:
# === TU TURNO ===
PLANTILLA_FLOJA = """Eres un asistente de atencion al cliente atento y servicial de Pacific-Blue.
Responde a la consulta de forma educada y formal, se util y cordial usando la siguiente informacion.

INFORMACION:
{contexto}

PREGUNTA: {pregunta}

Respuesta:"""     # <-- REEMPLAZA: pídele que responda de forma útil y amable,
                  #     con {contexto} y {pregunta}, y SIN prohibirle inventar

for p in ["puedo pagar con factura a 30 dias?",
          "hacen envios a Peru?",
          "cual es el descuento por volumen?"]:
    print(f"\n### {p}")
    print(f"  estricta: {responder_mia(p)}")
    print(f"  amable  : {responder_mia(p, plantilla=PLANTILLA_FLOJA)}")


### puedo pagar con factura a 30 dias?
  estricta: No se con los documentos que tengo.
  amable  : Lo siento, pero no tengo información específica sobre cómo realizar pagos con facturas a 30 días en Pacific Blue. Los plazos de pago pueden variar dependiendo de tu contrato o acuerdo específico con Pacific Blue. Te recomendaría que contactes directamente con su departamento de atención al cliente para obtener más detalles sobre este tema.

### hacen envios a Peru?
  estricta: No se menciona información específica sobre envíos a Perú en el contexto proporcionado.
  amable  : Sí, podemos realizar envíos a Perú. Sin embargo, es importante tener en cuenta que los envíos a destinos fuera de nuestra región (la Región del Biobío) requieren una carga mínima para no pagar recargos adicionales. Por favor, asegúrate de cumplir con estos requisitos antes de enviar cualquier artículo.

### cual es el descuento por volumen?
  estricta: No se con los documentos que tengo.
  amable  : Lo siento, pero p

**Mira la respuesta de la plantilla amable a los envíos a Perú.**

En la corrida de referencia el modelo no solo inventó el envío: lo respaldó con el **recargo del 40 %**, que sí está en el tarifario. Un dato verdadero pegado a una afirmación falsa, y el resultado es más creíble que una mentira entera.

Nada de la tubería cambió. Ni el índice, ni el troceo, ni el modelo, ni el k. Cambiaron dos frases del prompt.

Y ojo con la tercera pregunta: la plantilla amable **a veces sí contesta bien**. Un modo de falla intermitente es peor de administrar que uno constante.

---
## Bloque 6 — La versión empaquetada, y tus documentos

Todo lo del bloque 4 existe empaquetado. Ahora que sabes qué hay adentro, mira lo que ocupa:

In [98]:
# === DEMO — lo mismo, en una línea ===
from sentence_transformers import util

v = codificador_demo.encode("query: cuantos rollos de malla P50 hay disponibles",
                            convert_to_tensor=True, normalize_embeddings=True)
resultado = util.semantic_search(v, indice_mio, top_k=4)[0]

for r in resultado:
    print(f"{r['score']:.4f}  {trozos_mios[r['corpus_id']][:88]}")

0.9320  Ficha Tecnica - Malla P50. Existencias. Al cierre del mes hay 320 rollos disponibles en 
0.9128  Ficha Tecnica - Malla P50. Peso. 18 kilogramos por rollo.
0.9111  Ficha Tecnica - Malla P80. Existencias. Al cierre del mes hay 45 rollos disponibles en e
0.9103  Ficha Tecnica - Malla P50. Dimensiones. Rollo de 25 metros de largo por 1,50 metros de a


Una línea reemplaza a tu bloque 4 completo. **Y ahora puedes decir exactamente qué hace**, que es la diferencia entre usar una herramienta y depender de ella.

**LangChain** y **LlamaIndex** son el escalón siguiente: agregan cargadores de PDF, troceadores, conectores a bases vectoriales y memoria de conversación. Ninguno de los tres decide nada que tú no hayas decidido a mano hoy — y el troceo, que es lo que más movió tus números, lo siguen dejando en tus manos.

**TU TURNO — el último**

Carga **tus tres o cuatro documentos**, los que trajiste de tu proyecto, en un diccionario con el mismo formato de `CORPUS_DEMO`. Después corre las mismas funciones.

Dos condiciones para que el troceo funcione: cada documento parte con una **línea de título** y sus párrafos van separados por **una línea en blanco**.

> **Antes de subir nada:** los documentos van a un servicio de Google. Si son confidenciales, pide permiso o usa una versión sin datos reales.

> **¿No trajiste documentos?** No te quedes mirando. Sirve cualquier texto con estructura: el reglamento de la carrera, las bases de una licitación, el manual de un electrodoméstico. Lo que importa es que puedas juzgar si la respuesta está bien.

In [99]:
# === TU TURNO ===
MI_CORPUS = {
    "reglamento_academico": """Reglamento Academico de Pregrado - Informatica

Inscripcion de Asignaturas. Cada estudiante puede inscribir un maximo de 6 asignaturas por semestre academico regular.
No se permite exceder esta carga sin autorizacion del Director de carrera.
Requisitos de Asignaturas. Para cursar cualquier asignatura con prerrequisitos, es condicion obligatoria haber aprobado previamente todas las materias exigidas en la malla curricular oficial.
No existe inscripcion en paralelo de prerrequisitos pero existen casos excepcionales.
Asistencia y Aprobacion. La asistencia minima exigida para cursos practicos y laboratorios es del 75 por ciento.
La nota minima de aprobacion final es 4,0 en escala de 1,0 a 7,0.""",

    "ficha_malla_curricular": """Malla Curricular - Ingenieria Civil en Informatica

Linea de Programacion. La asignatura Introduccion a la Programacion es prerrequisito directo de Programacion Orientada a Objetos.
Esta ultima es requisito obligatorio para cursar Estructura de Datos.
Linea de Software y Datos. Bases de Datos se dicta en el tercer semestre y es prerrequisito para Ingenieria de Software.
Algoritmos y Complejidad requiere Estructura de Datos aprobada.
Practica Profesional. La primera practica profesional se realiza al completar el sexto semestre academico y contar con la Practicas Industrial realizada y Aprobada.""",

    "politica_titulacion": """Politica y Proceso de Titulacion

Modalidades de Titulacion. Los estudiantes pueden optar entre Proyecto de Titulo tradicional, Tesis de Investigacion guiada o Examen de Grado de Conocimientos Relevantes.
Plazos de Entrega. El informe final de memoria o proyecto de titulo debe entregarse a la comision evaluadora con al menos 15 dias habiles de anticipacion a la fecha de defensa oral."""
}

mis_trozos = por_parrafo_con_titulo_mia(MI_CORPUS)
mi_indice = codificador_demo.encode(["passage: " + t for t in mis_trozos],
                                    convert_to_tensor=True, normalize_embeddings=True)
print(len(mis_trozos), "trozos de", len(MI_CORPUS), "documentos")

# la pregunta que tu chat debería poder responder
MI_PREGUNTA = "cual es el requisito obligatorio para cursar Estructura de Datos?"

v = codificador_demo.encode("query: " + MI_PREGUNTA, convert_to_tensor=True,
                            normalize_embeddings=True)
mejores = torch.topk(mi_indice @ v, min(4, len(mis_trozos)))
contexto = "\n\n".join(mis_trozos[int(i)] for i in mejores.indices)
print("\n=== lo recuperado ===")
for s, i in zip(mejores.values, mejores.indices):
    print(f"{float(s):.3f}  {mis_trozos[int(i)][:88]}")
print("\n=== la respuesta ===")
print(preguntarle_al_modelo(PLANTILLA_DEMO.format(contexto=contexto, pregunta=MI_PREGUNTA)))

5 trozos de 3 documentos

=== lo recuperado ===
0.895  Malla Curricular - Ingenieria Civil en Informatica. Linea de Programacion. La asignatura
0.843  Reglamento Academico de Pregrado - Informatica. Inscripcion de Asignaturas. Cada estudia
0.822  Politica y Proceso de Titulacion. Politica y Proceso de Titulacion Modalidades de Titula
0.819  Reglamento Academico de Pregrado - Informatica. Reglamento Academico de Pregrado - Infor

=== la respuesta ===
Estructura de Datos es un requisito obligatorio para cursar la línea de Software y Datos.


In [100]:
# === VERIFICACIÓN ===
assert len(MI_CORPUS) >= 1 and MI_PREGUNTA.strip(), "Falta tu corpus o tu pregunta"
assert len(mis_trozos) >= 2, "Tu corpus tiene que dar al menos dos trozos"
assert all(len(t) > 20 for t in mis_trozos), "Hay trozos vacíos o casi vacíos"
print(f"Correcto: {len(mis_trozos)} trozos tuyos indexados, y la tubería completa corriendo sobre ellos.")

Correcto: 5 trozos tuyos indexados, y la tubería completa corriendo sobre ellos.


---
## El cierre: la pregunta de Ana

Ésta es la pregunta de la lámina 51, la que abre la clase del proyecto Pacific-Blue.

Para rechazar bien ese pedido hay **dos motivos**:

- hay **320 rollos** y están pidiendo **400** — ficha técnica de la P50;
- el **sábado no es día hábil** — política de despacho.

Antes de mirar la respuesta, mira **qué llegó al contexto**.

In [101]:
# === DEMO — el recorrido completo de una pregunta real ===
pregunta_ana = "me pueden despachar 400 mallas P50 el sabado"

print("=== lo que se recuperó con k=4 ===")
for s, t in buscar_mia(pregunta_ana, k=4):
    print(f"{s:.3f}  {t[:88]}")

print("\n=== lo que respondió ===")
print(responder_mia(pregunta_ana))

=== lo que se recuperó con k=4 ===
0.877  Ficha Tecnica - Malla P50. Existencias. Al cierre del mes hay 320 rollos disponibles en 
0.866  Ficha Tecnica - Malla P80. Existencias. Al cierre del mes hay 45 rollos disponibles en e
0.857  Ficha Tecnica - Malla P50. Plazo de entrega. 72 horas habiles desde la confirmacion del 
0.855  Ficha Tecnica - Malla P50. Peso. 18 kilogramos por rollo.

=== lo que respondió ===
No se puede despachar 400 mallas P50 el sábado debido al plazo de entrega establecido de 72 horas habiles desde la confirmación del pedido.


### El párrafo del sábado no está ahí

Vuelve a leer los cuatro trozos recuperados. Están las existencias de la P50, las de la P80, el peso y el plazo de entrega.

**No está el párrafo que dice que los sábados no se despacha.** Nunca llegó al contexto, así que el modelo no tenía cómo saberlo.

La celda siguiente busca en qué posición quedó.

In [102]:
# === DEMO — ¿dónde quedó el párrafo del sábado? ===
todos = buscar_mia(pregunta_ana, k=len(trozos_mios))
for posicion, (s, t) in enumerate(todos, 1):
    if "No se realizan despachos" in t:
        print(f"posición {posicion} de {len(trozos_mios)}, con coseno {s:.3f}")
        print(f"  {t[:95]}")
        break
print(f"\nel primero de la lista tenía coseno {todos[0][0]:.3f}")

posición 11 de 26, con coseno 0.837
  Politica de Despacho 2026 - Pacific-Blue. Dias de operacion. El centro de distribucion opera de

el primero de la lista tenía coseno 0.877


**Posición 11 de 26, con coseno 0,837 contra el 0,877 del primero: cuarenta milésimas.**

Está indexado, es el párrafo correcto, y con `k=4` no entra. Con `k=8` tampoco. Por la tabla del bloque 4, esta pregunta necesita **k=12** — y esa es la que costaba 565 tokens en vez de 149.

Ahora mira tu respuesta otra vez:

- Si el modelo rechazó **por el stock**, usó lo único que tenía y acertó por el motivo disponible.
- Si rechazó **por el plazo de 72 horas**, usó otro trozo y también suena razonable.
- Si mencionó **algo que no está en ningún documento** —una capacidad de carga, un mínimo de compra—, lo inventó. En nuestra corrida pasó exactamente eso.

Lo que ninguna versión puede hacer es rechazar por el sábado. **Ese dato no estaba, y nada en la tubería avisa que faltaba.**

> **Para responder en la celda de abajo:** el coseno hizo su trabajo, el troceo estaba bien, el prompt estaba bien escrito y la respuesta igual salió incompleta. **¿Qué componente tendría que haber cachado que faltaba el sábado?** ¿Y con qué información habría podido hacerlo — información que no está en ningún documento del corpus?

*(La respuesta la viste ayer. La vas a construir en el Laboratorio 2.)*

**Mi respuesta:**

*(Tenia que haber un auditor logico en ASP o un agente antes de responder. La pregunta junta dos cosas: el stock de las mallas y despachar un sabado. El coseno se fue solo por el producto y dejo el sabado afuera porque falta cruzar los datos con un calendario externo para saber que el sabado no es dia habil)*

---
## El presupuesto de reloj

Esta celda no enseña nada del RAG. Está para que sepamos **cuánto cuesta este cuaderno en una T4**, y para que quien lo dicte pueda planificar los 80 minutos con números y no con estimaciones.

In [103]:
# === DEMO — cuánto tardó cada cosa ===
print(f"descargar y cargar el codificador (471 MB) : {SEGUNDOS_CODIFICADOR:7.1f} s")
print(f"descargar y cargar el tokenizador          : {SEGUNDOS_TOKENIZADOR:7.1f} s")
print(f"descargar y cargar el generador (3,1 GB)   : {SEGUNDOS_GENERADOR:7.1f} s")

if SEGUNDOS_GENERACION:
    orden = sorted(SEGUNDOS_GENERACION)
    print(f"\n{len(SEGUNDOS_GENERACION)} respuestas generadas")
    print(f"  total                : {sum(SEGUNDOS_GENERACION):7.1f} s")
    print(f"  la más rápida        : {orden[0]:7.1f} s")
    print(f"  la mediana           : {orden[len(orden)//2]:7.1f} s")
    print(f"  la más lenta         : {orden[-1]:7.1f} s")

print(f"\ncuaderno completo, de punta a punta: {(time.time() - T_INICIO)/60:.1f} min")
print("(incluye las descargas, que la segunda vez no se pagan)")

descargar y cargar el codificador (471 MB) :     8.3 s
descargar y cargar el tokenizador          :     1.4 s
descargar y cargar el generador (3,1 GB)   :    15.0 s

13 respuestas generadas
  total                :    52.7 s
  la más rápida        :     0.6 s
  la mediana           :     3.2 s
  la más lenta         :    11.8 s

cuaderno completo, de punta a punta: 1.9 min
(incluye las descargas, que la segunda vez no se pagan)


---
## Lo que te llevas

| Lo que mediste | El número |
|---|---|
| Recuperar sin compartir ninguna palabra | coseno 0,875 y 0,827 |
| Partir por párrafo, sin el título | 5 de 9 |
| Pegarle el título — una línea | 9 de 9 |
| Documento entero: mismo acierto, más contexto | 829 tokens contra 219 |
| P50 contra P80 en la misma pregunta | 21 milésimas de diferencia |
| Preguntas que necesitan dos trozos, con k=4 | 2 de 5 |
| El párrafo del sábado en la pregunta de Ana | posición 11 de 26 |
| Cambiar dos frases del prompt | de negarse a inventar un envío a Perú |

**La frase para llevarse:** el modelo de lenguaje es el último eslabón y el que menos decidió. Lo que decidió fue el troceo.

### El Laboratorio 2, de trabajo personal

Los tres modos de falla de la lámina 40, provocados a propósito, sobre este mismo corpus más un documento nuevo: **un instructivo de 2024 que contradice a la política vigente y que nadie marcó como obsoleto**. Y encima, el auditor en ASP de las láminas 46 a 52 — que es lo que le faltaba a la respuesta de Ana.